# 08 — Evaluation: Fine-Tuned Model vs Gold Standard

Loads `Volcanex/baseline-schema-qwen3-vl-lora` (the trained LoRA adapter) on top of
Qwen3-VL-8B-Instruct, runs inference on random examples from the eval set, and shows
predicted JSON-LD side-by-side with the gold standard for visual inspection.

**Run on RunPod A100** — same setup as notebook 07.


In [ ]:
import os, sys, json, random, torch
from pathlib import Path
from dotenv import load_dotenv

# ── Environment ───────────────────────────────────────────────────────────────
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

PROJECT_ROOT = Path('/workspace/schema')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env', override=True)

HF_TOKEN   = os.getenv('HF_TOKEN')
MODELS_DIR = Path('/workspace/models')

# Compatibility shim for bitsandbytes on some PyTorch builds
if not hasattr(torch.nn.Module, 'set_submodule'):
    def _set_submodule(self, target, module):
        parent, _, last = target.rpartition('.')
        parent_mod = self.get_submodule(parent) if parent else self
        setattr(parent_mod, last, module)
    torch.nn.Module.set_submodule = _set_submodule

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB')


## Load Model + LoRA Adapter


In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL  = 'Qwen/Qwen3-VL-8B-Instruct'
ADAPTER_HF  = 'Volcanex/baseline-schema-qwen3-vl-lora'
ADAPTER_SUB = 'final'   # use the end-of-epoch adapter

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

try:
    import flash_attn  # noqa
    attn_impl = 'flash_attention_2'
except ImportError:
    attn_impl = 'sdpa'

print(f'Loading base model ({BASE_MODEL})...')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl,
    device_map='auto',
    token=HF_TOKEN,
    cache_dir=str(MODELS_DIR),
)
model.eval()

print(f'Loading LoRA adapter ({ADAPTER_HF}/{ADAPTER_SUB})...')
model = PeftModel.from_pretrained(model, ADAPTER_HF, subfolder=ADAPTER_SUB, token=HF_TOKEN)
model.eval()

processor = AutoProcessor.from_pretrained(
    BASE_MODEL, token=HF_TOKEN, cache_dir=str(MODELS_DIR),
    min_pixels=256*28*28, max_pixels=640*28*28,
)

print(f'✓ Ready. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')


## Inference Helper


In [ ]:
from qwen_vl_utils import process_vision_info

def generate_schema(messages, max_new_tokens=600):
    """Run inference on one eval example. Returns predicted JSON-LD string."""
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=img_inputs or None,
        return_tensors='pt',
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,       # greedy — deterministic for eval
            temperature=None,
            top_p=None,
            use_cache=True,
        )

    # Strip the prompt tokens, decode only what the model generated
    new_tokens = out_ids[0][inputs['input_ids'].shape[1]:]
    return processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('generate_schema() ready')


## 🎲 Random Eval — Visual Comparison


In [ ]:
from IPython.display import display, Image as IPImage, HTML
import base64

# ── Load eval set ─────────────────────────────────────────────────────────────
eval_path = PROJECT_ROOT / 'data' / 'eval.jsonl'
eval_examples = [json.loads(l) for l in open(eval_path)]
print(f'Eval examples: {len(eval_examples)}')

# ── Config ────────────────────────────────────────────────────────────────────
N_EXAMPLES = 5      # ← change this to see more/fewer examples
SEED       = 42     # ← change for different random picks

random.seed(SEED)
sample = random.sample(eval_examples, N_EXAMPLES)

# ── Display helper ────────────────────────────────────────────────────────────
def fmt_json(s):
    """Pretty-print a JSON string, or return raw if invalid."""
    try:
        return json.dumps(json.loads(s), indent=2, ensure_ascii=False)
    except Exception:
        return s

def show_comparison(ex, pred, idx):
    messages = ex['messages']
    gold = next(m['content'] for m in messages if m['role'] == 'assistant')
    img_path = None
    for m in messages:
        if m['role'] == 'user':
            for item in m.get('content', []):
                if isinstance(item, dict) and item.get('type') == 'image':
                    img_path = PROJECT_ROOT / item['image']

    # Parse types for quick headline
    try:
        pred_type = json.loads(pred).get('@type', '?')
    except Exception:
        pred_type = '⚠ invalid JSON'
    try:
        gold_type = json.loads(gold).get('@type', '?')
    except Exception:
        gold_type = '?'

    match_icon = '✅' if pred_type == gold_type else '❌'
    site_id = ex.get('id', f'example-{idx}')

    print(f'\n{"="*70}')
    print(f'  Example {idx+1}/{N_EXAMPLES} — {site_id}')
    print(f'  Gold @type: {gold_type}   Predicted @type: {pred_type}  {match_icon}')
    print(f'{"="*70}')

    # Screenshot
    if img_path and img_path.exists():
        display(IPImage(filename=str(img_path), width=500))
    else:
        print(f'  [screenshot not found: {img_path}]')

    # Side-by-side JSON comparison
    display(HTML(f"""
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:8px 0;font-size:13px">
      <div>
        <b style="color:#2a7a2a">✅ Gold Standard</b>
        <pre style="background:#f0fff0;border:1px solid #ccc;padding:10px;
                    border-radius:4px;overflow:auto;max-height:400px">{fmt_json(gold)}</pre>
      </div>
      <div>
        <b style="color:#1a4fa8">🤖 Model Prediction</b>
        <pre style="background:#f0f4ff;border:1px solid #ccc;padding:10px;
                    border-radius:4px;overflow:auto;max-height:400px">{fmt_json(pred)}</pre>
      </div>
    </div>
    """))

# ── Run ───────────────────────────────────────────────────────────────────────
print(f'Running inference on {N_EXAMPLES} random examples (SEED={SEED})...\n')

results = []
for i, ex in enumerate(sample):
    print(f'  [{i+1}/{N_EXAMPLES}] generating...', end=' ', flush=True)
    pred = generate_schema(ex['messages'])
    print('done')
    results.append({'ex': ex, 'pred': pred})

print('\n── Results ──────────────────────────────────────────────')
for i, r in enumerate(results):
    show_comparison(r['ex'], r['pred'], i)


## 📊 Batch Metrics (optional — runs on 50 examples)


In [ ]:
import numpy as np

RUN_BATCH_METRICS = False   # ← set True to run (~15 min on A100)
N_BATCH           = 50

def extract_properties(jsonld_str):
    try:
        obj = json.loads(jsonld_str)
        return {k for k in obj if k not in {'@context','@type','@id','@graph'}}
    except Exception:
        return set()

def property_f1(pred_str, gold_str):
    pred = extract_properties(pred_str)
    gold = extract_properties(gold_str)
    if not gold:
        return 0.0
    tp = len(pred & gold)
    p  = tp / len(pred) if pred else 0
    r  = tp / len(gold)
    return 2*p*r/(p+r) if (p+r) else 0.0

def type_match(pred_str, gold_str):
    try:
        pt = json.loads(pred_str).get('@type')
        gt = json.loads(gold_str).get('@type')
        ps = {pt} if isinstance(pt, str) else set(pt or [])
        gs = {gt} if isinstance(gt, str) else set(gt or [])
        return bool(ps & gs)
    except Exception:
        return False

def is_valid_json(s):
    try:
        json.loads(s); return True
    except Exception:
        return False

if RUN_BATCH_METRICS:
    batch = random.sample(eval_examples, N_BATCH)
    validity, type_acc, pf1s = [], [], []

    for i, ex in enumerate(batch):
        messages = ex['messages']
        gold = next(m['content'] for m in messages if m['role'] == 'assistant')
        print(f'  [{i+1}/{N_BATCH}]', end='\r')
        pred = generate_schema(messages)
        validity.append(int(is_valid_json(pred)))
        type_acc.append(int(type_match(pred, gold)))
        pf1s.append(property_f1(pred, gold))

    print(f'\n{"─"*40}')
    print(f'  N                : {N_BATCH}')
    print(f'  JSON validity    : {np.mean(validity)*100:.1f}%')
    print(f'  @type accuracy   : {np.mean(type_acc)*100:.1f}%')
    print(f'  Property F1      : {np.mean(pf1s):.3f}')
    print(f'{"─"*40}')
else:
    print('RUN_BATCH_METRICS=False — set to True and re-run this cell for full metrics.')
